In [11]:
!pip install yfinance pandas
import pandas as pd
import numpy as  np


In [12]:
import yfinance as yf

nifty50_tickers = [
    "ADANIENT.NS", "ADANIPORTS.NS", "APOLLOHOSP.NS", "ASIANPAINT.NS", "AXISBANK.NS",
    "BAJAJ-AUTO.NS", "BAJFINANCE.NS", "BAJAJFINSV.NS", "BEL.NS", "BHARTIARTL.NS",
    "BPCL.NS", "BRITANNIA.NS", "CIPLA.NS", "COALINDIA.NS", "DRREDDY.NS",
    "EICHERMOT.NS", "ETERNAL.NS", "GRASIM.NS", "HCLTECH.NS", "HDFCBANK.NS",
    "HDFCLIFE.NS", "HEROMOTOCO.NS", "HINDALCO.NS", "HINDUNILVR.NS", "ICICIBANK.NS",
    "ITC.NS", "INDUSINDBK.NS", "INFY.NS", "INDIGO.NS", "JSWSTEEL.NS",
    "JIOFIN.NS", "KOTAKBANK.NS", "LT.NS", "M&M.NS", "MARUTI.NS",
    "MAXHEALTH.NS", "NTPC.NS", "NESTLEIND.NS", "ONGC.NS", "POWERGRID.NS",
    "RELIANCE.NS", "SBILIFE.NS", "SHRIRAMFIN.NS", "SBIN.NS", "SUNPHARMA.NS",
    "TCS.NS", "TATACONSUM.NS", "TMPV.NS", "TATASTEEL.NS", "TECHM.NS",
    "TITAN.NS", "TRENT.NS", "ULTRACEMCO.NS", "WIPRO.NS"
]

In [13]:
Data=yf.download(
    tickers=nifty50_tickers,period="2y",interval="1d",
    auto_adjust="True"
)
Data.to_csv("nifty50_2year_data.csv")


[*********************100%***********************]  54 of 54 completed


In [14]:
df=pd.DataFrame(Data)

print(df.columns)

MultiIndex([( 'Close',   'ADANIENT.NS'),
            ( 'Close', 'ADANIPORTS.NS'),
            ( 'Close', 'APOLLOHOSP.NS'),
            ( 'Close', 'ASIANPAINT.NS'),
            ( 'Close',   'AXISBANK.NS'),
            ( 'Close', 'BAJAJ-AUTO.NS'),
            ( 'Close', 'BAJAJFINSV.NS'),
            ( 'Close', 'BAJFINANCE.NS'),
            ( 'Close',        'BEL.NS'),
            ( 'Close', 'BHARTIARTL.NS'),
            ...
            ('Volume',  'SUNPHARMA.NS'),
            ('Volume', 'TATACONSUM.NS'),
            ('Volume',  'TATASTEEL.NS'),
            ('Volume',        'TCS.NS'),
            ('Volume',      'TECHM.NS'),
            ('Volume',      'TITAN.NS'),
            ('Volume',       'TMPV.NS'),
            ('Volume',      'TRENT.NS'),
            ('Volume', 'ULTRACEMCO.NS'),
            ('Volume',      'WIPRO.NS')],
           names=['Price', 'Ticker'], length=270)


In [15]:
df.shape

(497, 270)

In [16]:
close_prices=df.xs('Close',level='Price',axis=1)
def get_consecutive_nan_count(col):
    is_nan = col.isnull()
    streak_id = (is_nan != is_nan.shift()).cumsum()
    nan_streaks = is_nan[is_nan].groupby(streak_id).size()
    return nan_streaks.max() if not nan_streaks.empty else 0

max_streaks = close_prices.apply(get_consecutive_nan_count)

tickers_to_drop = max_streaks[max_streaks > 5].index.tolist()
print(f"Dropping: {tickers_to_drop}")


Dropping: ['TMPV.NS']


In [17]:
clean_df = df.drop(columns=tickers_to_drop, level=1)
clean_df = clean_df.ffill()
clean_df.shape


(497, 265)

In [19]:
import yfinance as yf
usd_inr = yf.download('INR=X', start='2024-05-13', end='2026-05-11')
usd_inr = usd_inr[['Close']].rename(columns={'Close': 'USD_INR'})
df_forex=pd.DataFrame(usd_inr)
df_forex.head()


[*********************100%***********************]  1 of 1 completed


Price,USD_INR
Ticker,INR=X
Date,
2024-05-13,83.524803
2024-05-14,83.505501
2024-05-15,83.524200
2024-05-16,83.405098
2024-05-17,83.470100


In [20]:
!pip install pandas-datareader
import pandas_datareader.data as web
from datetime import datetime, timedelta
gdp = web.DataReader('GDP', 'fred', '2024-01-01', '2026-01-01')


gdp_monthly = (
    gdp
    .resample('MS')
    .ffill()
)
gdp_monthly = gdp_monthly.loc['2024-01-01':]
gdp_monthly.head()

,GDP
DATE,
2024-01-01,28708.161
2024-02-01,28708.161
2024-03-01,28708.161
2024-04-01,29147.044
2024-05-01,29147.044


In [21]:
cpi = web.DataReader('CPIAUCSL', 'fred', '2024-01-01', '2026-01-01')

cpi_monthly = (
    cpi
    .resample('MS')
    .ffill()
)
cpi.head()

,CPIAUCSL
DATE,
2024-01-01,309.698
2024-02-01,310.967
2024-03-01,312.345
2024-04-01,313.023
2024-05-01,313.175


In [22]:
wpi = web.DataReader('PPIACO', 'fred','2024-01-01', '2026-01-01')

wpi_monthly = (
    wpi
    .resample('MS')
    .ffill()
)
wpi.head()

,PPIACO
DATE,
2024-01-01,251.306
2024-02-01,254.926
2024-03-01,255.095
2024-04-01,256.978
2024-05-01,255.313


In [25]:
repo = pd.read_csv("repo_rate.csv")


repo.columns = ['DATE', 'REPO_RATE']


repo['DATE'] = pd.to_datetime(repo['DATE'])


repo.set_index('DATE', inplace=True)


repo_monthly = repo.resample('MS').ffill()

print(repo_monthly.head())


            REPO_RATE
DATE                 
2024-02-01        NaN
2024-03-01        6.5
2024-04-01        6.5
2024-05-01        6.5
2024-06-01        6.5


C:\Users\vichu\AppData\Local\Temp\ipykernel_9916\1635157001.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  repo['DATE'] = pd.to_datetime(repo['DATE'])


In [26]:
macro = (
    cpi_monthly
    .join(gdp_monthly, how='inner')
    .join(wpi_monthly, how='inner')
    .join(repo_monthly, how='inner')
)

macro.columns = ['CPI', 'GDP', 'WPI', 'REPO_RATE']
macro.reset_index()
macro.index=pd.to_datetime(macro.index)
macro_daily=macro.resample('D').ffill()
macro_daily.head()

,CPI,GDP,WPI,REPO_RATE
DATE,,,,
2024-02-01,310.967,28708.161,254.926,NaN
2024-02-02,310.967,28708.161,254.926,NaN
2024-02-03,310.967,28708.161,254.926,NaN
2024-02-04,310.967,28708.161,254.926,NaN
2024-02-05,310.967,28708.161,254.926,NaN


In [35]:

import pandas as pd
import os

stock_folder = "data"

output_folder = "final_data"

os.makedirs(output_folder, exist_ok=True)



for file in os.listdir(stock_folder):

    if file.endswith(".csv"):
        file_path = os.path.join(stock_folder, file)
        stock_df = pd.read_csv(file_path)
        stock_df['Date'] = pd.to_datetime(stock_df['Date'])
        stock_df.set_index('Date', inplace=True)
        merged_df = pd.merge(
            stock_df,
            macro_daily,
            left_index=True,
            right_index=True,
            how='left'
        )
        merged_df = merged_df.ffill().bfill()
        output_path = os.path.join(output_folder, file)
        merged_df.to_csv(output_path)
        print(f"Merged file saved: {output_path}")

Merged file saved: final_data\ADANIENT.NS.csv
Merged file saved: final_data\ADANIPORTS.NS.csv
Merged file saved: final_data\APOLLOHOSP.NS.csv
Merged file saved: final_data\ASIANPAINT.NS.csv
Merged file saved: final_data\AXISBANK.NS.csv
Merged file saved: final_data\BAJAJ-AUTO.NS.csv
Merged file saved: final_data\BAJAJFINSV.NS.csv
Merged file saved: final_data\BAJFINANCE.NS.csv
Merged file saved: final_data\BEL.NS.csv
Merged file saved: final_data\BHARTIARTL.NS.csv
Merged file saved: final_data\BPCL.NS.csv
Merged file saved: final_data\BRITANNIA.NS.csv
Merged file saved: final_data\CIPLA.NS.csv
Merged file saved: final_data\COALINDIA.NS.csv
Merged file saved: final_data\DRREDDY.NS.csv
Merged file saved: final_data\EICHERMOT.NS.csv
Merged file saved: final_data\ETERNAL.NS.csv
Merged file saved: final_data\GRASIM.NS.csv
Merged file saved: final_data\HCLTECH.NS.csv
Merged file saved: final_data\HDFCBANK.NS.csv
Merged file saved: final_data\HDFCLIFE.NS.csv
Merged file saved: final_data\HERO

In [36]:
macro_daily.to_csv(
    "macro_daily.csv",
    index=True
)

print("macro_daily.csv saved successfully!")

macro_daily.csv saved successfully!
